In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from core.nn.timesfm import NewsTimesFM_2p5_Model
from transformers import AutoTokenizer
from safetensors.torch import load_file
import torch
import polars as pl
from core.training.data.dataset import TimesFMDataset, collate_fn
from clearml import InputModel
from core.nn.text_encoder.tokenizer import NewsTokenizerWrapper
from torch.utils.data import DataLoader
from core.training.data import TimesFMDataModule
from core.training.configs import DataConfig
from functools import partial
import plotly.express as px
import random
from clearml import Task, OutputModel
from transformers.models.modernbert.modeling_modernbert import ModernBertModel
import orjson
from pathlib import Path
from core.nn.timesfm.configs import ModernBertConfig, TimesFM_2p5_200M_Config

In [3]:
data_cfg = DataConfig(
    dataset_id="a9dc8daf2d89450b8812ae4cf78514a1",
    num_workers=8,
    batch_size=3,
    output_patch_len=20,
    input_patch_len=32,
    context_len=256,
)

In [5]:
datamodule = TimesFMDataModule(
    data_cfg=data_cfg,
    tokenizer_path=InputModel(
        model_id="c84c36cfe62b41e7b4c3c9217beff551"
    ).get_local_copy(),
)

In [6]:
datamodule.prepare_data()
datamodule.setup(stage="fit")

In [6]:
(data['inputs_ts'][..., -1].log().unsqueeze(-1) + data['targets_log_returns'].cumsum(-1)).exp()

NameError: name 'data' is not defined

In [7]:
train_loader = datamodule.train_dataloader()

In [8]:
dl = iter(train_loader)

In [9]:
batch = next(dl)
batch.keys()


dict_keys(['inputs_ts', 'inputs_log_returns', 'mask_ts', 'targets_log_returns', 'targets', 'inputs_text', 'mask_text'])

In [11]:
(batch["inputs_ts"][..., -1].log().unsqueeze(-1) + batch["targets_log_returns"].cumsum(-1)).exp()

tensor([[[334.0000, 333.6999, 334.2000, 335.0499, 335.7499, 335.4499, 335.9999,
          338.7499, 338.4501, 339.2500, 338.9000, 340.5999, 340.9000, 343.0000,
          342.4999, 343.1999, 342.3999, 344.4499, 344.6999, 344.9499],
         [349.3000, 349.5500, 324.3500, 318.2000, 319.3500, 324.0000, 323.7000,
          323.7000, 322.2001, 320.5500, 317.1000, 316.0500, 317.1499, 316.4500,
          316.2000, 319.3500, 316.8500, 318.1499, 317.9999, 316.1000],
         [323.0000, 323.3000, 324.2000, 327.4501, 329.8501, 330.4000, 331.1501,
          329.3500, 329.1001, 330.1500, 330.4001, 328.1001, 329.1001, 328.0500,
          328.6000, 329.2500, 328.8500, 330.3500, 333.5501, 332.6500],
         [328.8500, 327.3500, 327.7000, 327.5499, 328.4999, 328.8999, 319.0000,
          318.0500, 317.1499, 319.5499, 318.7999, 319.4000, 319.0999, 319.2498,
          316.8499, 316.2499, 316.2499, 313.9500, 312.8999, 312.1999],
         [312.5000, 312.8000, 310.1001, 305.2001, 307.4502, 305.5001, 306.05

In [10]:
news_timesfm = NewsTimesFM_2p5_Model.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models/new-timesfm-2p5-small-bert",
    dtype=torch.bfloat16,
)

In [11]:
inputs_ts = batch["inputs_log_returns"].to(torch.bfloat16)
mask_ts = batch["mask_ts"].to(torch.bfloat16)
targets = batch["targets_log_returns"].to(torch.bfloat16)

inputs_text = batch["inputs_text"]
mask_text = batch["mask_text"]

output = news_timesfm.forecast(
    inputs_ts=inputs_ts,
    mask_ts=mask_ts,
    inputs_text=inputs_text,
    mask_text=mask_text,
    targets=targets,
)
output

{'normalized_outputs': tensor([[[ 3.0273e-01,  3.6328e-01,  4.3359e-01,  4.7852e-01,  3.2227e-01,
            2.8320e-01,  2.4707e-01,  2.7539e-01,  2.8125e-01,  2.9297e-01,
            3.4375e-01,  3.6328e-01,  3.5352e-01,  4.1797e-01,  8.2520e-02,
            1.5918e-01,  3.0859e-01,  3.3984e-01,  4.0039e-01,  3.1055e-01],
          [ 3.9258e-01,  4.6484e-01,  5.2344e-01,  5.6250e-01,  4.3945e-01,
            4.2578e-01,  3.4961e-01,  3.8672e-01,  3.9648e-01,  4.1016e-01,
            4.3359e-01,  4.6094e-01,  4.7070e-01,  5.2344e-01,  1.6602e-01,
            2.3730e-01,  3.7891e-01,  4.1016e-01,  4.5898e-01,  3.7891e-01],
          [ 1.8750e-01,  2.7539e-01,  3.6719e-01,  3.9258e-01,  3.0273e-01,
            2.7930e-01,  1.4844e-01,  2.2168e-01,  2.0215e-01,  1.8164e-01,
            2.3145e-01,  2.7344e-01,  2.5000e-01,  3.0469e-01,  2.0142e-02,
            1.0645e-01,  2.7539e-01,  3.4375e-01,  3.4375e-01,  2.3828e-01],
          [ 2.1094e-01,  3.0078e-01,  3.6133e-01,  4.0039e-01, 

In [15]:
output["outputs"].std()

tensor(0.0044, dtype=torch.bfloat16, grad_fn=<StdBackward0>)

In [19]:
calc_pred = (batch["inputs_ts"][..., -1].log().unsqueeze(-1) + output["outputs"].cumsum(-1)).exp()

In [14]:
batch["targets"]

tensor([[[1091.4000, 1109.0000, 1132.0000, 1116.2000, 1117.8000, 1105.4000,
          1107.2000, 1115.0000, 1139.0000, 1128.0000, 1147.2000, 1147.2000,
          1131.8000, 1144.0000, 1159.4000, 1146.6000, 1188.8000, 1163.0000,
          1165.8000, 1170.2000],
         [1374.4000, 1337.8000, 1323.2000, 1343.2000, 1334.2000, 1326.4000,
          1369.2000, 1351.2000, 1332.8000, 1315.6000, 1367.6000, 1344.8000,
          1347.8000, 1340.4000, 1340.0000, 1344.4000, 1320.4000, 1313.0000,
          1316.2000, 1280.6000],
         [1287.8000, 1301.0000, 1293.0000, 1280.8000, 1308.4000, 1311.8000,
          1334.6000, 1304.4000, 1341.0000, 1343.6000, 1366.8000, 1360.0000,
          1375.4000, 1333.0000, 1339.0000, 1315.6000, 1334.4000, 1361.0000,
          1408.0000, 1436.2000],
         [1509.2000, 1528.2000, 1507.8000, 1503.0000, 1521.8000, 1553.0000,
          1537.8000, 1607.8000, 1632.2000, 1611.8000, 1668.6000, 1768.8000,
          1735.6000, 1749.0000, 1827.8000, 1874.8000, 1881.2000, 

In [27]:
batch_size = calc_pred.size(0)
batch_idx = random.randint(0, batch_size - 1)
last_pred = calc_pred[batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()
last_target = (
    batch["targets"][batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()
)

fig = px.line()
fig.add_scatter(y=last_pred, mode="lines+markers", name="Prediction")
fig.add_scatter(y=last_target, mode="lines+markers", name="Target")
fig.show()